In [1]:
import pandas as pd

In [2]:
parts = [
    "../data/split/part_1.csv",
    "../data/split/part_2.csv",
    "../data/split/part_3.csv",
    "../data/split/part_4.csv"
]
samples = []
for part in parts:
    sample = pd.read_csv(part, usecols=["owner", "name", "combined_text"])
    samples.append(sample)

full_df = pd.concat(samples, ignore_index=True)
df_20k = full_df.sample(n=20_000, random_state=42).reset_index(drop=True)
print(df_20k.shape)  # exactly (20000, 3)

(20000, 3)


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_20k = TfidfVectorizer(max_features=20000, min_df=5, max_df=0.8)
data_matrix = tfidf_20k.fit_transform(df_20k['combined_text'])  # fit_transform, not transform, since this is a NEW vectorizer for this small sample

In [6]:
data_matrix.shape

(20000, 4791)

In [16]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=3000,random_state=42)
reduced_matrix = svd.fit_transform(data_matrix)

In [17]:
reduced_matrix.shape

(20000, 3000)

In [19]:
print(f"Explained variance: {svd.explained_variance_ratio_.sum():.4f}") #This tells you how much information were captured into the compressed version

Explained variance: 0.9380


In [20]:
import faiss
import numpy as np

vectors = reduced_matrix.astype(np.float32)
faiss.normalize_L2(vectors)

In [21]:
d = vectors.shape[1]
nlist = 1000

quantizer = faiss.IndexFlatIP(d)
index = faiss.IndexIVFFlat(quantizer,d,nlist,faiss.METRIC_INNER_PRODUCT)

In [22]:
index.train(vectors)
index.add(vectors)

In [23]:
index.nprobe = 10

In [24]:
import pandas as pd
indices = pd.Series(df_20k.index, index=df_20k['name']).drop_duplicates()

In [25]:
def search_repos(query_text, top_n=5):
    #  vectorize the query using the SAME tfidf vectorizer
    query_tfidf = tfidf_20k.transform([query_text])
    
    # reduce using the SAME fitted SVD model
    query_reduced = svd.transform(query_tfidf).astype(np.float32)

    faiss.normalize_L2(query_reduced)

    D, I = index.search(query_reduced, top_n)
    
    return df_20k[['owner', 'name']].iloc[I[0]]

In [26]:
print(search_repos("hospital management using python or js"))

                     owner                    name
7111   Sudhanshu-kumar-927         Data-Management
11919           sunilj0lly  engineering-management
10264                dskst               librarian
4208      The-SCRIPT-Group                   Hades
3021        emimontesdeoca                   Lovis


In [30]:
df_20k["combined_text"].iloc[11919]

'engineering-management My approach and thoughts on effective engineering management  leadership management learning'

In [31]:
matches = df_20k[df_20k['combined_text'].str.contains('hospital', case=False, na=False)]
print(matches.shape)
print(matches[['owner', 'name']])

(4, 3)
               owner                                      name
11387  wiseweb-works            react-hospital-appointment-app
13786       jkovacic                        coursera_imageproc
14618         66deep  TielingHeHospitalPatientManagementSystem
18013        MrNewaz                      Global-Hospital-v2.0
